In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_lineage_analysis/query_history"
tgt_silver_table = "data_governance.silver_lineage_analysis.lineage_query_history"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:
# transofrmations and cleaning
df = df.withColumn("compute_type", col("compute.type")) \
       .withColumn("cluster_id", col("compute.cluster_id")) \
       .withColumn("warehouse_id", col("compute.warehouse_id"))


df = df.withColumn("job_id", col("query_source.job_info.job_id")) \
       .withColumn("job_run_id", col("query_source.job_info.job_run_id")) \
       .withColumn("job_task_run_id", col("query_source.job_info.job_task_run_id")) \
       .withColumn("notebook_id", col("query_source.notebook_id")) \
       .withColumn("dashboard_id", col("query_source.dashboard_id")) \
       .withColumn("pipeline_id", col("query_source.pipeline_info.pipeline_id"))


df = df.withColumn("event_date", to_date("start_time")) \
       .withColumn("event_year", year("start_time")) \
       .withColumn("event_month", month("start_time")) \
       .withColumn("event_day", dayofmonth("start_time")) \
       .withColumn("event_hour", hour("start_time"))


df = df.withColumn("statement_type", lower(trim(col("statement_type"))))
df = df.withColumn(
    "query_category",
    when(col("statement_type").isin("create", "drop", "alter", "truncate"), "DDL")
    .when(col("statement_type").isin("insert", "update", "delete"), "DML")
    .when(col("statement_type").isin("commit", "savepoint", "rollback"), "TCL")
    .when(col("statement_type").isin("select"), "DQL")
    .when(col("statement_type").isin("grant", "revoke"), "DCL")
    .otherwise("OTHER")
)


df = df.withColumn(
    "query_origin",
    when(col("notebook_id").isNotNull(),"NOTEBOOK")
    .when(col("job_id").isNotNull(),"JOB")
    .when(col("dashboard_id").isNotNull(),"DASHBOARD")
    .when(col("pipeline_id").isNotNull(),"PIPELINE")
    .otherwise("OTHER")
)

df = df.withColumn(
    "query_duration_sec",
    col("total_duration_ms")/1000
)

df = df.withColumn(
    "compute_wait_ratio",
    col("waiting_for_compute_duration_ms") / col("total_duration_ms")
)

df = df.withColumn("read_mb", col("read_bytes")/1024/1024) \
       .withColumn("written_mb", col("written_bytes")/1024/1024) \
       .withColumn("shuffle_read_mb", col("shuffle_read_bytes")/1024/1024)

df = df.withColumn(
    "is_failed_query",
    when(col("execution_status")!="FINISHED",True).otherwise(False)
)

df = df.dropDuplicates(["workspace_id","statement_id"])

df = df.fillna({
    "read_rows":0,
    "written_rows":0,
    "read_bytes":0,
    "written_bytes":0
})

df = df.drop("compute", "query_source")

In [0]:
df.display()

In [0]:
df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("event_year", "event_month", "event_day") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_lineage_analysis.lineage_query_history;